# LDOS(omega) 谱检查

读取 `spectra_ldos.csv` 或 `processed_ldos.csv`，快速查看站点-频率热图和指定频率的空间分布。

In [ ]:
using CSV
using DataFrames
using Plots

analysis_dir = get(ENV, "DWAVEHMC_ANALYSIS_DIR", @__DIR__)
csv_path = begin
    p1 = joinpath(analysis_dir, "spectra_ldos.csv")
    p2 = joinpath(analysis_dir, "processed_ldos.csv")
    isfile(p1) ? p1 : p2
end

df = CSV.read(csv_path, DataFrame)
first(df, 5)

In [ ]:
sites = sort(unique(df.site))
omegas = sort(unique(df.omega))

ldos_site_omega = zeros(length(sites), length(omegas))
for (is, site) in pairs(sites), (iw, omega) in pairs(omegas)
    rows = df[(df.site .== site) .& (df.omega .== omega), :]
    ldos_site_omega[is, iw] = rows.LDOS[1]
end

heatmap(omegas, sites, ldos_site_omega;
        xlabel="omega", ylabel="site", colorbar_title="LDOS",
        title="LDOS(site, omega)")

In [ ]:
omega_target = 0.0
omega_selected = omegas[argmin(abs.(omegas .- omega_target))]
slice = df[df.omega .== omega_selected, :]

Lx = maximum(slice.x)
Ly = maximum(slice.y)
ldos_xy = fill(NaN, Ly, Lx)
for row in eachrow(slice)
    ldos_xy[row.y, row.x] = row.LDOS
end

heatmap(1:Lx, 1:Ly, ldos_xy;
        xlabel="x", ylabel="y", aspect_ratio=:equal,
        colorbar_title="LDOS",
        title="LDOS at omega=$(round(omega_selected, digits=6))")

In [ ]:
site_to_plot = first(sites)
site_df = df[df.site .== site_to_plot, :]
sort!(site_df, :omega)

plot(site_df.omega, site_df.LDOS;
     ribbon=site_df.Error,
     xlabel="omega", ylabel="LDOS",
     label="site $(site_to_plot)",
     title="LDOS spectrum")